## Step 0

In [1]:
import warnings
warnings.filterwarnings("ignore")

import mlflow
import mlflow.sklearn
import numpy as np
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score

mlflow.set_tracking_uri("http://localhost:5000")
mlflow.set_experiment("mnist-mlp-classifier")
print("Tracking URI:", mlflow.get_tracking_uri())

Tracking URI: http://localhost:5000


## Step 1 — Run 6+ Hyperparameter Experiments

In [2]:
X, y = load_digits(return_X_y=True)
X = X / 16.0
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

def train_and_evaluate(hidden_layer_sizes=(100,), learning_rate=0.001, alpha=0.0001, max_epochs=100):
    from sklearn.metrics import log_loss
    
    # Split validation set from training
    X_tr, X_val, y_tr, y_val = train_test_split(X_train, y_train, test_size=0.2, random_state=42)
    
    model = MLPClassifier(
        hidden_layer_sizes=hidden_layer_sizes,
        learning_rate_init=learning_rate,
        alpha=alpha,
        max_iter=1,
        warm_start=True,
        random_state=42,
    )
    
    train_losses = []
    val_accs = []
    
    for epoch in range(max_epochs):
        model.fit(X_tr, y_tr)
        train_loss = log_loss(y_tr, model.predict_proba(X_tr))
        val_preds = model.predict(X_val)
        val_acc = accuracy_score(y_val, val_preds)
        train_losses.append(train_loss)
        val_accs.append(val_acc)
    
    test_preds = model.predict(X_test)
    test_acc = accuracy_score(y_test, test_preds)
    
    return model, train_losses, val_accs, test_acc

_, _, _, _ = train_and_evaluate()
print("Data loaded. Ready for experiments.")

Data loaded. Ready for experiments.


In [6]:
def train_and_log(hidden_layer_sizes=(100,), learning_rate=0.001, alpha=0.0001, max_epochs=100, run_name=None):
    with mlflow.start_run(run_name=run_name):
        # --- Log hyperparameters ---
        mlflow.log_param("hidden_layer_sizes", str(hidden_layer_sizes))
        mlflow.log_param("learning_rate", learning_rate)
        mlflow.log_param("alpha", alpha)
        mlflow.log_param("max_epochs", max_epochs)
        
        # Train and evaluate with epoch-by-epoch history
        model, train_losses, val_accs, test_acc = train_and_evaluate(hidden_layer_sizes, learning_rate, alpha, max_epochs)
        
        # --- Log epoch-by-epoch metrics to MLflow ---
        for epoch, (train_loss, val_acc) in enumerate(zip(train_losses, val_accs)):
            mlflow.log_metric("train_loss", train_loss, step=epoch)
            mlflow.log_metric("val_accuracy", val_acc, step=epoch)
        
        # Log final test accuracy
        mlflow.log_metric("test_accuracy", test_acc)
        
        mlflow.set_tag("model", "MLP")
        mlflow.set_tag("dataset", "MNIST")
        
        run_id = mlflow.active_run().info.run_id
        print(f"✓ {run_name} | lr={learning_rate} | hidden={hidden_layer_sizes} | val_acc={val_accs[-1]:.4f}")
        return run_id

run_ids = []

rid1 = train_and_log(hidden_layer_sizes=(50,), learning_rate=0.0001, max_epochs=20, run_name="exp1-small-lowlr")
run_ids.append(rid1)

rid2 = train_and_log(hidden_layer_sizes=(50,), learning_rate=0.01, max_epochs=20, run_name="exp2-small-highlr")
run_ids.append(rid2)

rid3 = train_and_log(hidden_layer_sizes=(100,), learning_rate=0.0001, max_epochs=20, run_name="exp3-medium-lowlr")
run_ids.append(rid3)

rid4 = train_and_log(hidden_layer_sizes=(100,), learning_rate=0.01, max_epochs=20, run_name="exp4-medium-highlr")
run_ids.append(rid4)

rid5 = train_and_log(hidden_layer_sizes=(200,), learning_rate=0.0001, max_epochs=20, run_name="exp5-large-lowlr")
run_ids.append(rid5)

rid6 = train_and_log(hidden_layer_sizes=(200,), learning_rate=0.01, max_epochs=20, run_name="exp6-large-highlr")
run_ids.append(rid6)

rid7 = train_and_log(hidden_layer_sizes=(100, 50), learning_rate=0.001, max_epochs=20, run_name="exp7-deep-medlr")
run_ids.append(rid7)

print("\n✓ All 7 experiments completed!")

✓ exp1-small-lowlr | lr=0.0001 | hidden=(50,) | val_acc=0.1701
🏃 View run exp1-small-lowlr at: http://localhost:5000/#/experiments/2/runs/01aa235c597041fbb760b9e3db68bbf7
🧪 View experiment at: http://localhost:5000/#/experiments/2
✓ exp2-small-highlr | lr=0.01 | hidden=(50,) | val_acc=0.9722
🏃 View run exp2-small-highlr at: http://localhost:5000/#/experiments/2/runs/6b79321384304c1e891f6e475283bd6c
🧪 View experiment at: http://localhost:5000/#/experiments/2
✓ exp3-medium-lowlr | lr=0.0001 | hidden=(100,) | val_acc=0.3542
🏃 View run exp3-medium-lowlr at: http://localhost:5000/#/experiments/2/runs/59c5825b36e74e9ca54342b692b0e175
🧪 View experiment at: http://localhost:5000/#/experiments/2
✓ exp4-medium-highlr | lr=0.01 | hidden=(100,) | val_acc=0.9653
🏃 View run exp4-medium-highlr at: http://localhost:5000/#/experiments/2/runs/f0ca2807fff8480c9307e484c005af7e
🧪 View experiment at: http://localhost:5000/#/experiments/2
✓ exp5-large-lowlr | lr=0.0001 | hidden=(200,) | val_acc=0.6424
🏃 View